In [1]:
import os
import cv2
import numpy as np
import pandas as pd

from skimage.feature import local_binary_pattern

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [2]:
train_path = r"D:\Seed Quality Prediction\data-20260503T170400Z-3-001\data\train"
val_path = r"D:\Seed Quality Prediction\data-20260503T170400Z-3-001\data\val"

In [3]:
def extract_features(image_path):

    img = cv2.imread(image_path)

    if img is None:
        return None

    img = cv2.resize(img, (128,128))

    # HSV Color Histogram
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    hist = cv2.calcHist(
        [hsv],
        [0,1,2],
        None,
        [8,8,8],
        [0,256,0,256,0,256]
    )

    hist = cv2.normalize(hist, hist).flatten()

    # LBP Texture
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    lbp = local_binary_pattern(
        gray,
        P=24,
        R=8,
        method='uniform'
    )

    lbp_hist,_ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0,27),
        range=(0,26)
    )

    lbp_hist = lbp_hist.astype("float")
    lbp_hist /= (lbp_hist.sum()+1e-7)

    return np.hstack([hist, lbp_hist])

In [4]:
def load_dataset(folder):

    X = []
    y = []

    for label in os.listdir(folder):

        class_path = os.path.join(folder, label)

        if not os.path.isdir(class_path):
            continue

        for img_name in os.listdir(class_path):

            img_path = os.path.join(class_path, img_name)

            feat = extract_features(img_path)

            if feat is not None:
                X.append(feat)
                y.append(label)

    return np.array(X), np.array(y)

In [5]:
X_train, y_train = load_dataset(train_path)
X_val, y_val = load_dataset(val_path)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)

Train: (14340, 538)
Validation: (3479, 538)


In [6]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [ ]:
param_grid = {
    'C':[10,50,100,200],
    'gamma':[0.01,0.001,'scale'],
    'kernel':['rbf']
}

grid = GridSearchCV(
    SVC(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)